# Combined team v14: Soup Kernel Challenge

Select **Settings → Accelerator → GPU T4 x2** and **Internet On**, then **Run All**.
The harness uses one GPU. Keep other GPU notebooks stopped. No GitHub token is needed.
This notebook uses a new unique checkout and preserves previous run folders.

Vladimir contributed the unpadded model path; Nurzhigit/Codex added parallel
vocabulary reductions and, for v14, compact dropout. v13 has a completed
participant-supplied 10/10 console report. **This is the v14 mask repair. The original fd5e482 failed exact mask tests; this repair awaits GPU validation.**
Expect roughly 8–12 minutes, including two pairs and quality evaluation.


In [ ]:
REPO_URL = "https://github.com/promaladoi/Decentrathon-6.0.git"
COMMIT = "3016d555b15f6625d61f0c19806c916b128e5b1a"
VERSION = "v14"


In [ ]:
import subprocess, os, json, zipfile, uuid
from pathlib import Path
from IPython.display import display, FileLink
print(subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True))
subprocess.run(['python','-m','pip','install','-q','soup-cli[train]==0.77.0','transformers==5.16.1','peft==0.20.0','trl==0.29.1','bitsandbytes==0.50.2','accelerate==1.14.0','pytest'],check=True)


In [ ]:
# Export the existing v13 run if it is still available, without retraining.
old=Path('/kaggle/working/nurzhigit_v13')
old_out=old/'runs_codex_v13'
if all((old_out/n).exists() for n in ('result.json','checks.json')):
    with zipfile.ZipFile('/kaggle/working/codex_v13.zip','w',zipfile.ZIP_DEFLATED) as z:
        for n in ('result.json','checks.json'): z.write(old_out/n,n)
        for f in (old/'solution').rglob('*.py'): z.write(f,str(f.relative_to(old)))
        for n in ('adapter_config.json','adapter_model.safetensors'):
            f=old_out/'solution_1/adapter'/n
            if f.exists(): z.write(f,'adapter/'+n)
    display(FileLink('codex_v13.zip'))
else:
    print('No previous v13 artifacts in this session; continuing with the pinned experiment.')


In [ ]:
assert not subprocess.check_output(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader'],text=True).strip(), 'GPU is busy: stop the other GPU work before running this notebook.'
p=Path('/kaggle/working')/('nurzhigit_'+VERSION+'_'+uuid.uuid4().hex[:8])
subprocess.run(['git','clone','-q',REPO_URL,str(p)],check=True,timeout=120)
subprocess.run(['git','-C',str(p),'checkout','-q',COMMIT],check=True)
print('MEASURED CODE:',subprocess.check_output(['git','-C',str(p),'rev-parse','HEAD'],text=True).strip())
subprocess.run(['python','task/prepare_data.py'],cwd=p,check=True)


In [ ]:
# A failed correctness test stops before benchmarking.
subprocess.run(['python','-m','pytest','solution','-q'],cwd=p,check=True)
subprocess.run(['python','-m','pytest','tests','-q'],cwd=p,env=dict(os.environ,SOUP_APPLY_SOLUTION='1'),check=True)


In [ ]:
out='runs_codex_'+VERSION
subprocess.run(['python','harness/run_task.py','--repeats','2','--out',out],cwd=p,check=True)
subprocess.run(['python','harness/check_rules.py','--result',out+'/result.json'],cwd=p)
subprocess.run(['python','harness/score.py',out+'/result.json'],cwd=p)


In [ ]:
archive='codex_'+VERSION+'.zip'
with zipfile.ZipFile('/kaggle/working/'+archive,'w',zipfile.ZIP_DEFLATED) as z:
    for n in ('result.json','checks.json'): z.write(p/out/n,n)
    for f in (p/'solution').rglob('*.py'): z.write(f,str(f.relative_to(p)))
    for n in ('adapter_config.json','adapter_model.safetensors'):
        z.write(p/out/'solution_1/adapter'/n,'adapter/'+n)
print('Download the archive below to preserve exact reports, code and adapter.')
display(FileLink(archive))


Save a version only after reviewing all ten checks. Keep the exact commit pinned.
For final submission, preserve the executed output, raw reports and the team-selected adapter.
